### Setup 

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter0_fundamentals"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import torchinfo
except:
    %pip install torchinfo jaxtyping einops datasets

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import einops
import torch as t
import torchinfo
import wandb
from datasets import load_dataset
from einops.layers.torch import Rearrange
from jaxtyping import Float, Int
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from tqdm import tqdm

# Make sure exercises are in the path
chapter = "chapter0_fundamentals"
section = "part5_vaes_and_gans"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))


import part5_vaes_and_gans.tests as tests
import part5_vaes_and_gans.utils as utils
from part2_cnns.utils import print_param_count
from plotly_utils import imshow

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
# device = t.device("cpu")


from part2_cnns.solutions import BatchNorm2d, Conv2d, Linear, ReLU, Sequential, Flatten

from part5_vaes_and_gans.solutions import ConvTranspose2d
from part5_vaes_and_gans.solutions import DCGAN as SolutionDCGAN

### Helpers

In [ ]:
def get_dataset(dataset: Literal["MNIST", "CELEB"], train: bool = True) -> Dataset:
    assert dataset in ["MNIST", "CELEB"]

    if dataset == "CELEB":
        image_size = 64
        assert train, "CelebA dataset only has a training set"
        transform = transforms.Compose(
            [
                transforms.Resize(image_size),
                transforms.CenterCrop(image_size),
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ]
        )
        trainset = datasets.ImageFolder(root=exercises_dir / "part5_vaes_and_gans/data/celeba", transform=transform)

    elif dataset == "MNIST":
        img_size = 28
        transform = transforms.Compose(
            [transforms.Resize(img_size), transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
        )
        trainset = datasets.MNIST(
            root=exercises_dir / "part5_vaes_and_gans/data",
            transform=transform,
            download=True,
        )

    return trainset

In [ ]:
def display_data(x: Tensor, nrows: int, title: str):
    """Displays a batch of data, using plotly."""
    ncols = x.shape[0] // nrows
    # Reshape into the right shape for plotting (make it 2D if image is monochrome)
    y = einops.rearrange(x, "(b1 b2) c h w -> (b1 h) (b2 w) c", b1=nrows).squeeze()
    # Normalize in the 0-1 range, then map to integer type
    y = (y - y.min()) / (y.max() - y.min())
    y = (y * 255).to(dtype=t.uint8)
    # Display data
    imshow(
        y,
        binary_string=(y.ndim == 2),
        height=50 * (nrows + 4),
        width=50 * (ncols + 5),
        title=f"{title}<br>single input shape = {x[0].shape}",
    )


trainset_mnist = get_dataset("MNIST")
trainset_celeb = get_dataset("CELEB")

# Display MNIST
x = next(iter(DataLoader(trainset_mnist, batch_size=25)))[0]
display_data(x, nrows=5, title="MNIST data")

# Display CelebA
x = next(iter(DataLoader(trainset_celeb, batch_size=25)))[0]
display_data(x, nrows=5, title="CelebA data")

In [ ]:
class Tanh(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        return (x.exp() - t.exp(-x)) / (x.exp() + t.exp(-x))


class LeakyReLU(nn.Module):
    def __init__(self, negative_slope: float = 0.01):
        super().__init__()
        self.negative_slope = negative_slope

    def forward(self, x: Tensor) -> Tensor:
        sloped = x * self.negative_slope
        return t.where(x > 0, x, sloped)

    def extra_repr(self) -> str:
        return f"negative_slope={self.negative_slope}"


class Sigmoid(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        return 1/(1 + t.exp(-x))



tests.test_Tanh(Tanh)
tests.test_LeakyReLU(LeakyReLU)
tests.test_Sigmoid(Sigmoid)

In [ ]:
def initialize_weights(model: nn.Module) -> None:
    """
    Initializes weights according to the DCGAN paper (details at the end of page 3 of the DCGAN paper), by modifying the
    weights of the model in place.
    """
    # Previous version wasn't working - trying solution
    for module in model.modules():
        # print(f'initializing {module}')
        if isinstance(module, (ConvTranspose2d, Conv2d, Linear)):
            nn.init.normal_(module.weight.data, 0.0, 0.02)
        elif isinstance(module, BatchNorm2d):
            nn.init.normal_(module.weight.data, 1.0, 0.02)
            nn.init.constant_(module.bias.data, 0.0)


tests.test_initialize_weights(
    initialize_weights, ConvTranspose2d, Conv2d, Linear, BatchNorm2d
)

### Real GAN stuff

#### GAN

In [ ]:
class Generator(nn.Module):
    def __init__(
        self,
        latent_dim_size: int = 100,
        img_size: int = 64,
        img_channels: int = 3,
        hidden_channels: list[int] = [128, 256, 512],
    ):
        """
        Implements the generator architecture from the DCGAN paper (the diagram at the top
        of page 4). We assume the size of the activations doubles at each layer (so image
        size has to be divisible by 2 ** len(hidden_channels)).

        Args:
            latent_dim_size:
                the size of the latent dimension, i.e. the input to the generator
            img_size:
                the size of the image, i.e. the output of the generator
            img_channels:
                the number of channels in the image (3 for RGB, 1 for grayscale)
            hidden_channels:
                the number of channels in the hidden layers of the generator (starting closest
                to the middle of the DCGAN and going outward, i.e. in chronological order for
                the generator)
        """
        n_layers = len(hidden_channels)
        assert (
            img_size % (2**n_layers) == 0
        ), "activation size must double at each layer"

        super().__init__()

        # Looks like this is in reverse order?
        hidden_channels = list(reversed(hidden_channels))
        starting_size = int(img_size / (2**n_layers))

        # First linear + reshape latent space into first hidden channel
        self.project_and_reshape = nn.Sequential(
            Linear(
                latent_dim_size,
                hidden_channels[0] * starting_size * starting_size,
                bias=False,
            ),
            Rearrange(
                "batch (c w h) -> batch c w h",
                c=hidden_channels[0],
                w=starting_size,
                h=starting_size,
            ),
            BatchNorm2d(hidden_channels[0]),
            ReLU(),
        )

        # Hidden layers
        def block(in_channels, out_channels):
            return [
                ConvTranspose2d(in_channels, out_channels, 4, 2, 1),
                BatchNorm2d(out_channels),
                ReLU(),
            ]

        blocks = [
            layer
            for i, channels in enumerate(hidden_channels[:-1])
            for layer in block(channels, hidden_channels[i + 1])
        ]
        last_layer = [
            ConvTranspose2d(hidden_channels[-1], img_channels, 4, 2, 1),
            Tanh(),
        ]
        self.hidden_layers = nn.Sequential(*blocks, *last_layer)


    def forward(self, x: Tensor) -> Tensor:
        x = self.project_and_reshape(x)
        x = self.hidden_layers(x)
        return x


class Discriminator(nn.Module):
    def __init__(
        self,
        img_size: int = 64,
        img_channels: int = 3,
        hidden_channels: list[int] = [128, 256, 512],
    ):
        """
        Implements the discriminator architecture from the DCGAN paper (the mirror image of
        the diagram at the top of page 4). We assume the size of the activations doubles at
        each layer (so image size has to be divisible by 2 ** len(hidden_channels)).

        Args:
            img_size:
                the size of the image, i.e. the input of the discriminator
            img_channels:
                the number of channels in the image (3 for RGB, 1 for grayscale)
            hidden_channels:
                the number of channels in the hidden layers of the discriminator (starting
                closest to the middle of the DCGAN and going outward, i.e. in reverse-
                chronological order for the discriminator)
        """
        n_layers = len(hidden_channels)
        assert (
            img_size % (2**n_layers) == 0
        ), "activation size must double at each layer"

        super().__init__()

        ending_size = int(img_size / (2**n_layers))

        def block(in_channels, out_channels):
            return [
                Conv2d(in_channels, out_channels, 4, 2, 1),
                BatchNorm2d(out_channels),
                LeakyReLU(),
            ]

        blocks = [
            layer
            for i, channels in enumerate(hidden_channels[:-1])
            for layer in block(channels, hidden_channels[i + 1])
        ]
        first_layer = [Conv2d(img_channels, hidden_channels[0], 4, 2, 1), LeakyReLU()]
        self.hidden_layers = nn.Sequential(*first_layer, *blocks)

        # Create latent space vector
        self.classifier = nn.Sequential(
            Rearrange(
                "batch c w h -> batch (c w h)",
            ),
            Linear(
                hidden_channels[-1] * ending_size * ending_size,
                1,
                bias=False,
            ),
            Sigmoid(),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.hidden_layers(x)
        x = self.classifier(x)
        return x.squeeze()  # remove dummy `out_channels` dimension


test = Discriminator()


class DCGAN(nn.Module):
    netD: Discriminator
    netG: Generator

    def __init__(
        self,
        latent_dim_size: int = 100,
        img_size: int = 64,
        img_channels: int = 3,
        hidden_channels: list[int] = [128, 256, 512],
    ):
        super().__init__()
        self.latent_dim_size = latent_dim_size
        self.img_size = img_size
        self.img_channels = img_channels
        self.hidden_channels = hidden_channels
        self.netD = Discriminator(img_size, img_channels, hidden_channels)
        self.netG = Generator(latent_dim_size, img_size, img_channels, hidden_channels)

        # Init weights
        initialize_weights(self.netD)
        initialize_weights(self.netG)

test = DCGAN()
 
# print_param_count(Generator(), SolutionDCGAN().netG)
# print_param_count(Discriminator(), SolutionDCGAN().netD)

# model = DCGAN().to(device)
# x = t.randn(3, 100).to(device)
# print(torchinfo.summary(model.netG, input_data=x), end="\n\n")
# print(torchinfo.summary(model.netD, input_data=model.netG(x)))

In [ ]:
# Make sure the weights are init correctly
test_GAN = DCGAN()

for param in test_GAN.netD.parameters():
    print(param.mean())
    print(param.std())



#### Trainer

In [ ]:
@dataclass
class DCGANArgs:
    """
    Class for the arguments to the DCGAN (training and architecture).
    Note, we use field(defaultfactory(...)) when our default value is a mutable object.
    """

    # architecture
    latent_dim_size: int = 100
    hidden_channels: list[int] = field(default_factory=lambda: [128, 256, 512])

    # data & training
    dataset: Literal["MNIST", "CELEB"] = "CELEB"
    batch_size: int = 64
    epochs: int = 3
    lr: float = 0.0002
    betas: tuple[float, float] = (0.5, 0.999)
    clip_grad_norm: float | None = 1.0

    # logging
    use_wandb: bool = False
    wandb_project: str | None = "day5-gan"
    wandb_name: str | None = None
    log_every_n_steps: int = 250


class DCGANTrainer:
    def __init__(self, args: DCGANArgs):
        self.args = args
        self.trainset = get_dataset(self.args.dataset)
        self.trainloader = DataLoader(
            self.trainset, batch_size=args.batch_size, shuffle=True, num_workers=8
        )

        batch, img_channels, img_height, img_width = next(iter(self.trainloader))[
            0
        ].shape
        assert img_height == img_width

        self.model = (
            DCGAN(args.latent_dim_size, img_height, img_channels, args.hidden_channels)
            .to(device)
            .train()
        )
        self.optG = t.optim.Adam(
            self.model.netG.parameters(), lr=1e-3, betas=args.betas
        )
        # Trying to lower learning rate of discrim to allow generator more breathing space
        self.optD = t.optim.Adam(
            self.model.netD.parameters(), lr=1e-5, betas=args.betas
        ) 

    def training_step_discriminator(
        self,
        img_real: Float[Tensor, "batch channels height width"],
        img_fake: Float[Tensor, "batch channels height width"],
    ) -> Float[Tensor, ""]:
        """
        Generates a real and fake image, and performs a gradient step on the discriminator to maximize
        log(D(x)) + log(1-D(G(z))). Logs to wandb if enabled.
        """
        # Zero gradients
        self.optD.zero_grad()

        # Calculate D(x) and D(G(z)), for use in the objective function
        D_x = self.model.netD(img_real)
        D_G_z = self.model.netD(img_fake)

        # Calculate loss
        # lossD = -(t.log(D_x).mean() + t.log(1 - D_G_z).mean())
        # Same idea as below - new loss that clamps andis more stable
        loss_real = nn.BCELoss()(D_x, t.ones_like(D_x))
        loss_fake = nn.BCELoss()(D_G_z, t.zeros_like(D_G_z))
        lossD = loss_real + loss_fake
        
        # print(f"{lossD=}")

        # Gradient descent step (with optional clipping)
        lossD.backward()
        if self.args.clip_grad_norm is not None:
            nn.utils.clip_grad_norm_(
                self.model.netD.parameters(), self.args.clip_grad_norm
            )
        self.optD.step()

        if self.args.use_wandb:
            wandb.log(dict(lossD=lossD), step=self.step)
        return lossD

    def training_step_generator(
        self, img_fake: Float[Tensor, "batch channels height width"]
    ) -> Float[Tensor, ""]:
        """
        Performs a gradient step on the generator to maximize log(D(G(z))). Logs to wandb if enabled.
        """
        # Zero gradients
        self.optG.zero_grad()

        # Calculate D(G(z)), for use in the objective function
        D_G_z = self.model.netD(img_fake)

        # Calculate loss

        # Trying new loss - clamping
        # lossG = -(t.log(D_G_z).mean())
        lossG = nn.BCELoss()(D_G_z, t.ones_like(D_G_z))
        # print(f"{lossG=}")

        # Gradient descent step (with optional clipping)
        lossG.backward()
        if self.args.clip_grad_norm is not None:
            nn.utils.clip_grad_norm_(
                self.model.netG.parameters(), self.args.clip_grad_norm
            )
        self.optG.step()

        if self.args.use_wandb:
            wandb.log(dict(lossG=lossG), step=self.step)
        return lossG
    
    @t.inference_mode()
    def log_samples(self) -> None:
        """
        Performs evaluation by generating 8 instances of random noise and passing them through the generator, then
        optionally logging the results to Weights & Biases.
        """
        assert (
            self.step > 0
        ), "First call should come after a training step. Remember to increment `self.step`."
        self.model.netG.eval()

        # Generate random noise
        t.manual_seed(42)
        noise = t.randn(10, self.model.latent_dim_size).to(device)
        # Get generator output
        output = self.model.netG(noise)
        # Clip values to make the visualization clearer
        output = output.clamp(output.quantile(0.01), output.quantile(0.99))
        # Log to weights and biases
        if self.args.use_wandb:
            output = einops.rearrange(output, "b c h w -> b h w c").cpu().numpy()
            wandb.log({"images": [wandb.Image(arr) for arr in output]}, step=self.step)
        else:
            display_data(output, nrows=1, title="Generator-produced images")

        self.model.netG.train()

    def train(self) -> DCGAN:
        """Performs a full training run."""
        self.step = 0
        if self.args.use_wandb:
            wandb.init(project=self.args.wandb_project, name=self.args.wandb_name)

        for epoch in range(self.args.epochs):
            progress_bar = tqdm(
                self.trainloader, total=len(self.trainloader), ascii=True
            )

            for img_real, label in progress_bar:
                # Generate random noise & fake image
                noise = t.randn(self.args.batch_size, self.args.latent_dim_size).to(
                    device
                )
                img_real = img_real.to(device)
                img_fake = self.model.netG(noise)

                # Training steps
                lossD = self.training_step_discriminator(img_real, img_fake.detach())

                # Try training the generator twice
                lossG1 = self.training_step_generator(img_fake)

                # Regenerate to train:
                noise2 = t.randn(self.args.batch_size, self.args.latent_dim_size).to(
                    device
                )
                img_fake2 = self.model.netG(noise2)
                lossG2 = self.training_step_generator(img_fake2)

                # Update progress bar
                self.step += 1
                progress_bar.set_description(
                    f"{epoch=}, {lossD=:.4f}, {lossG1=:.4f}, {lossG2=:.4f}, batches={self.step}"
                )

                # Log batch of data
                if self.step % self.args.log_every_n_steps == 0:
                    self.log_samples()

        if self.args.use_wandb:
            wandb.finish()

        return self.model

#### Training

In [ ]:
print(device)

In [ ]:
# Arguments for CelebA
args = DCGANArgs(
    dataset="CELEB",
    hidden_channels=[128, 256, 512],
    batch_size=12,  # if you get OOM errors, reduce this!
    epochs=5,
    use_wandb=False,
)
trainer = DCGANTrainer(args)
dcgan = trainer.train()